# Phase 2 & 3 â€” Preprocessing and Model Training

**Phase 2:** Data Preprocessing â€” scaling, handling imbalance with SMOTE

**Phase 3:** Training 4 ML models â€” Logistic Regression, Random Forest, XGBoost, Isolation Forest

In [5]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Imbalanced-learn
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# XGBoost
import xgboost as xgb

# Joblib for model persistence
import joblib

sys.path.append('../')
warnings.filterwarnings('ignore')
plt.style.use('dark_background')

print('All libraries loaded.')
print(f'XGBoost version: {xgb.__version__}')

All libraries loaded.
XGBoost version: 3.1.1


In [6]:
from src.preprocess import load_and_validate, scale_features, apply_smote, prepare_data

print('Preprocessing module imported successfully.')

Preprocessing module imported successfully.


In [8]:
# Load and validate the dataset
df = load_and_validate("../data/creditcard.csv")
print(f"Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")

# Scale Amount and Time features
# Note: scale_features() scales in-place — column names stay as Amount and Time
df_scaled = scale_features(df)

print("Before scaling — Amount stats:")
print(df[["Amount", "Time"]].describe())

print("After scaling — Amount & Time (scaled in-place):")
print(df_scaled[["Amount", "Time"]].describe())


INFO:src.preprocess:Loading dataset from: ../data/creditcard.csv
INFO:src.preprocess:No missing values found.
INFO:src.preprocess:Dataset loaded: 283726 rows | Fraud: 473 (0.1667%) | Legit: 283253
INFO:src.preprocess:Feature scaling applied: RobustScaler(Amount), StandardScaler(Time).


Dataset loaded: 283,726 rows x 31 columns
Before scaling — Amount stats:
              Amount           Time
count  283726.000000  283726.000000
mean       88.472687   94811.077600
std       250.399437   47481.047891
min         0.000000       0.000000
25%         5.600000   54204.750000
50%        22.000000   84692.500000
75%        77.510000  139298.000000
max     25691.160000  172792.000000
After scaling — Amount & Time (scaled in-place):
              Amount          Time
count  283726.000000  2.837260e+05
mean        0.924387  1.218105e-16
std         3.482123  1.000002e+00
min        -0.305938 -1.996823e+00
25%        -0.228063 -8.552128e-01
50%         0.000000 -2.131081e-01
75%         0.771937  9.369423e-01
max       356.962314  1.642362e+00


## Handling Class Imbalance with SMOTE

**SMOTE** (Synthetic Minority Over-sampling Technique) creates synthetic fraud samples by interpolating between existing fraud samples and their k-nearest neighbors.

**Why not just duplicate fraud samples?** Simple duplication causes overfitting â€” the model memorizes specific fraud transactions rather than learning general fraud patterns.

**SMOTE process:**
1. For each fraud sample, find k nearest fraud neighbors
2. Randomly pick one neighbor
3. Create a new synthetic point along the line segment between them
4. Repeat until desired balance is achieved

In [9]:
# Prepare train/test splits with SMOTE applied to training set
X_train, X_test, y_train, y_test = prepare_data('../data/creditcard.csv')

print('Dataset split complete:')
print(f'  X_train shape: {X_train.shape}')
print(f'  X_test shape:  {X_test.shape}')
print(f'  y_train shape: {y_train.shape}')
print(f'  y_test shape:  {y_test.shape}')

print('\nClass distribution in training set (after SMOTE):')
unique, counts = np.unique(y_train, return_counts=True)
for cls, cnt in zip(unique, counts):
    label = 'Fraud' if cls == 1 else 'Legitimate'
    print(f'  Class {cls} ({label}): {cnt:,}')

print('\nClass distribution in test set (original, no SMOTE):')
unique, counts = np.unique(y_test, return_counts=True)
for cls, cnt in zip(unique, counts):
    label = 'Fraud' if cls == 1 else 'Legitimate'
    print(f'  Class {cls} ({label}): {cnt:,}')

INFO:src.preprocess:Loading dataset from: ../data/creditcard.csv
INFO:src.preprocess:No missing values found.
INFO:src.preprocess:Dataset loaded: 283726 rows | Fraud: 473 (0.1667%) | Legit: 283253
INFO:src.preprocess:Feature scaling applied: RobustScaler(Amount), StandardScaler(Time).
INFO:src.preprocess:Train size: 226980 | Test size: 56746
INFO:src.preprocess:Before SMOTE — Class distribution: {np.int64(0): np.int64(226602), np.int64(1): np.int64(378)}
  File "C:\Users\HP\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "C:\Users\HP\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\HP\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, pree

Dataset split complete:
  X_train shape: (453204, 30)
  X_test shape:  (56746, 30)
  y_train shape: (453204,)
  y_test shape:  (56746,)

Class distribution in training set (after SMOTE):
  Class 0 (Legitimate): 226,602
  Class 1 (Fraud): 226,602

Class distribution in test set (original, no SMOTE):
  Class 0 (Legitimate): 56,651
  Class 1 (Fraud): 95


## Model Training

We train 4 models with different paradigms:

| Model | Type | Strength |
|-------|------|----------|
| Logistic Regression | Linear baseline | Fast, interpretable |
| Random Forest | Ensemble (bagging) | Robust, handles non-linearity |
| XGBoost | Ensemble (boosting) | State-of-the-art performance |
| Isolation Forest | Anomaly detection | Unsupervised, no labels needed |

### Model 1: Logistic Regression (Baseline)

In [10]:
from src.models import train_logistic_regression

print('Training Logistic Regression...')
lr_model = train_logistic_regression(X_train, y_train)
print('Logistic Regression trained successfully!')

# Feature importance (coefficients)
feature_names = ['Time', 'Amount'] + [f'V{i}' for i in range(1, 29)]
coef_df = pd.DataFrame({
    'Feature': feature_names[:len(lr_model.coef_[0])],
    'Coefficient': lr_model.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False)

print('\nTop 10 most important features (by |coefficient|):')
print(coef_df.head(10).to_string(index=False))

INFO:src.models:Training Logistic Regression...


Training Logistic Regression...


INFO:src.models:Logistic Regression training complete.


Logistic Regression trained successfully!

Top 10 most important features (by |coefficient|):
Feature  Coefficient
    V14    -1.671645
    V10    -1.321225
    V12    -1.245808
    V17    -1.057824
    V20    -1.005282
     V4     0.903604
    V22     0.790925
     V1     0.747859
     V5     0.697379
    V11     0.667529


### Model 2: Random Forest

In [11]:
from src.models import train_random_forest

print('Training Random Forest (this may take 1-2 minutes)...')
rf_model = train_random_forest(X_train, y_train)
print('Random Forest trained successfully!')

# OOB Score (Out-Of-Bag)
if hasattr(rf_model, 'oob_score_'):
    print(f'\nOOB Score: {rf_model.oob_score_:.4f}')
    print('(OOB score is an unbiased estimate of model accuracy on unseen data)')

# Feature importance
feature_names = ['Time', 'Amount'] + [f'V{i}' for i in range(1, 29)]
fi_df = pd.DataFrame({
    'Feature': feature_names[:len(rf_model.feature_importances_)],
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print('\nTop 10 most important features (Random Forest):')
print(fi_df.head(10).to_string(index=False))

INFO:src.models:Training Random Forest (100 estimators)...


Training Random Forest (this may take 1-2 minutes)...


INFO:src.models:Random Forest training complete.


Random Forest trained successfully!

Top 10 most important features (Random Forest):
Feature  Importance
    V14    0.223883
    V12    0.151654
     V4    0.097470
    V10    0.092559
    V17    0.073078
    V11    0.063409
     V3    0.046028
     V7    0.029470
     V2    0.024913
    V16    0.023686


### Model 3: XGBoost

In [12]:
from src.models import train_xgboost

print('Training XGBoost...')
xgb_model = train_xgboost(X_train, y_train)
print('XGBoost trained successfully!')

# Feature importance
xgb_fi = xgb_model.feature_importances_
feature_names = ['Time', 'Amount'] + [f'V{i}' for i in range(1, 29)]
xgb_fi_df = pd.DataFrame({
    'Feature': feature_names[:len(xgb_fi)],
    'Importance': xgb_fi
}).sort_values('Importance', ascending=False)

print('\nTop 10 most important features (XGBoost):')
print(xgb_fi_df.head(10).to_string(index=False))

INFO:src.models:Training XGBoost...


Training XGBoost...


INFO:src.models:XGBoost training complete.


XGBoost trained successfully!

Top 10 most important features (XGBoost):
Feature  Importance
    V14    0.465924
    V10    0.148361
     V4    0.049139
     V8    0.030669
    V12    0.023661
    V17    0.020549
 Amount    0.017429
    V11    0.017423
    V18    0.016679
    V13    0.016001


### Model 4: Isolation Forest (Anomaly Detection)

In [14]:
from src.models import train_isolation_forest

print('Training Isolation Forest...')
iso_model = train_isolation_forest(X_train)
print('Isolation Forest trained successfully!')
print('\nIsolation Forest uses anomaly scores â€” no explicit feature importance.')
print(f'Contamination parameter: {iso_model.contamination}')
print('\nNote: Isolation Forest is unsupervised â€” it detects anomalies')
print('without using the fraud labels during training.')

INFO:src.models:Training Isolation Forest (anomaly detection)...


Training Isolation Forest...


INFO:src.models:Isolation Forest training complete.


Isolation Forest trained successfully!

Isolation Forest uses anomaly scores â€” no explicit feature importance.
Contamination parameter: 0.001727

Note: Isolation Forest is unsupervised â€” it detects anomalies
without using the fraud labels during training.


### Train All Models

Save all trained models to disk using joblib for later evaluation.

In [15]:
from src.models import train_all_models

os.makedirs('../models', exist_ok=True)

print('Training all models and saving to ../models/ ...')
models = train_all_models(X_train, y_train, '../models/')

print('\nAll models trained and saved:')
for model_name, model_obj in models.items():
    model_path = f'../models/{model_name}.pkl'
    size_kb = os.path.getsize(model_path) / 1024 if os.path.exists(model_path) else 0
    print(f'  {model_name:25s} -> {model_path} ({size_kb:.1f} KB)')

print('\nPhase 3 complete â€” Models ready for evaluation in Phase 4.')

INFO:src.models:Training Logistic Regression...


Training all models and saving to ../models/ ...


INFO:src.models:Logistic Regression training complete.
INFO:src.models:Saved logistic_regression.joblib
INFO:src.models:Training Random Forest (100 estimators)...
INFO:src.models:Random Forest training complete.
INFO:src.models:Saved random_forest.joblib
INFO:src.models:Training XGBoost...
INFO:src.models:XGBoost training complete.
INFO:src.models:Saved xgboost.joblib
INFO:src.models:Training Isolation Forest (anomaly detection)...
INFO:src.models:Isolation Forest training complete.
INFO:src.models:Saved isolation_forest.joblib
INFO:src.models:All models saved to: ..\models



All models trained and saved:
  logistic_regression       -> ../models/logistic_regression.pkl (1.5 KB)
  random_forest             -> ../models/random_forest.pkl (12164.4 KB)
  xgboost                   -> ../models/xgboost.pkl (0.0 KB)
  isolation_forest          -> ../models/isolation_forest.pkl (1713.5 KB)

Phase 3 complete â€” Models ready for evaluation in Phase 4.
